# 🕳️ TerraLens AI — Detecção de Buracos com YOLOv8

**Análise:** Reconhecimento de Buracos em Imagens 360°

**Pipeline:**
1. Download do dataset (Roboflow Universe)
2. Configuração do YOLOv8
3. Fine-tuning com dataset de buracos
4. Avaliação mAP50
5. Exportação para o backend TerraLens

**Datasets:** Pothole Detection (Roboflow), CrackForest

In [ ]:
# Instalação
# !pip install ultralytics roboflow

from ultralytics import YOLO
import os

print('✅ Ultralytics YOLOv8 carregado!')

## 1. Download do Dataset (Roboflow)

In [ ]:
# Opção A: Roboflow Universe (requer API key gratuita)
# from roboflow import Roboflow
# rf = Roboflow(api_key='SUA_API_KEY_AQUI')
# project = rf.workspace().project('pothole-detection-dataset')
# dataset = project.version(1).download('yolov8')

# Opção B: Dataset local
# Estrutura esperada:
# data/
#   train/
#     images/  *.jpg
#     labels/  *.txt  (formato YOLO)
#   val/
#     images/
#     labels/

DATASET_YAML = 'data/buracos.yaml'
print(f'Dataset: {DATASET_YAML}')

## 2. Criação do data.yaml

In [ ]:
# Cria o arquivo de configuração do dataset
yaml_content = """
path: ../../data/buracos
train: train/images
val: val/images

nc: 3
names:
  0: Buraco
  1: Trinca
  2: Deformacao
"""

os.makedirs('data', exist_ok=True)
with open(DATASET_YAML, 'w') as f:
    f.write(yaml_content)

print(f'✅ {DATASET_YAML} criado!')

## 3. Fine-tuning YOLOv8

In [ ]:
# Carrega modelo pré-treinado no COCO
model = YOLO('yolov8x.pt')  # yolov8n (rápido) | yolov8x (preciso)

# Fine-tuning
results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # GPU (0) ou 'cpu'
    project='runs/buracos',
    name='yolov8x_buracos',
    patience=20,       # early stopping
    save=True,
    plots=True,
)

print('✅ Treino concluído!')
print(f'Melhor modelo: {results.save_dir}/weights/best.pt')

## 4. Avaliação

In [ ]:
# Avalia no conjunto de validação
best_model = YOLO('runs/buracos/yolov8x_buracos/weights/best.pt')
metrics = best_model.val(data=DATASET_YAML)

print(f'\n📊 Métricas de Validação:')
print(f'   mAP50:    {metrics.box.map50:.4f}')
print(f'   mAP50-95: {metrics.box.map:.4f}')
print(f'   Precision: {metrics.box.mp:.4f}')
print(f'   Recall:    {metrics.box.mr:.4f}')

## 5. Exportar para o Backend

In [ ]:
import shutil

src = 'runs/buracos/yolov8x_buracos/weights/best.pt'
dst = '../../backend/models/view360/buracos/yolov8_buracos.pt'

os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(src, dst)

print(f'✅ Modelo exportado para: {dst}')
print(f'\n🚀 TerraLens AI agora detecta buracos de verdade!')